In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from src.constants import BASEDIR
os.chdir(BASEDIR)

from src.data.objects import Protein
from src.tools.sifts import SIFTS
from src.sequence.fasta import read_fasta_lines
from src.structure.sifts import build_protein_from_mapped_pdb_chain, backbone_xyz
from evcouplings.compare.pdb import PDB

We probably want some way of filtering for coverage (PDB coverage of uniprot or vv.)

Columns are mapped as follows:


"PDB": "pdb_id",  
"CHAIN": "pdb_chain",  
"SP_PRIMARY": "uniprot_ac",  
"RES_BEG": "resseq_start",  
"RES_END": "resseq_end",  
"PDB_BEG": "coord_start",  
"PDB_END": "coord_end",  
"SP_BEG": "uniprot_start",  
"SP_END": "uniprot_end",  

In [ ]:
sifts = SIFTS(sifts_table_file="data/evcouplings_sifts.csv")

In [ ]:
sifts.table

In [ ]:
res = sifts.by_uniprot_id("A4XEP2", reduce_chains=True)
res.hits

In [ ]:
res.mapping

In [ ]:
chain = PDB.from_id("9jdt").get_chain("A")
chain.residues

In [ ]:
len(chain.coords.residue_index.unique())

In [ ]:
c = chain.remap(res.mapping[res.hits.iloc[0]["mapping_index"]])
c.residues

In [ ]:
from evcouplings.compare.sifts import fetch_uniprot_mapping


def retrieve_uniprot_sequence(uniprot_id):
    uniprot_text = fetch_uniprot_mapping([uniprot_id])
    lines = read_fasta_lines(uniprot_text.text.split("\n"))
    label, seq = next(lines)
    return seq


seq = retrieve_uniprot_sequence("A4XEP2")
seq

In [ ]:
len(seq)

In [ ]:
for ix, row in c.residues.iterrows():
    if not seq[int(row["id"])-1] == row["one_letter_code"]:
        print(row["id"], row["one_letter_code"], seq[int(row["id"])-1])

In [ ]:
df = pd.read_parquet("data/foldseek_af50_struct/0.parquet")
accessions = set([acc for accessions in df['accessions'] for acc in accessions])

In [ ]:
sifts.table[sifts.table["uniprot_ac"].isin(accessions)]

In [ ]:
res = sifts.by_uniprot_id("Q5W0Q7", reduce_chains=True)
res.hits

In [ ]:
seq = retrieve_uniprot_sequence("Q5W0Q7")
len(seq)

In [ ]:
res.mapping

In [ ]:
chain = PDB.from_id("7zju").get_chain("A")

In [ ]:
chain.coords

In [ ]:
chain.coords

In [ ]:
chain.residues["xyz"] = backbone_xyz(chain)

In [ ]:
chain.residues.iloc[0]["xyz"]

In [ ]:
chain.residues

In [ ]:
pdb_prot = build_protein_from_mapped_pdb_chain(chain, seq)

In [ ]:
import numpy as np
import py3Dmol


has_coords_mask = ~np.isnan(pdb_prot.backbone_coords).any(axis=(1, 2))
prot = Protein.from_pdb(os.path.join(BASEDIR, "data/AF-Q5W0Q7-F1-model_v4.pdb"))
has_coords_indices = np.argwhere(has_coords_mask).reshape(-1)
view = py3Dmol.view(width=450, height=400)
prot.slice_arrays(has_coords_indices).view_superimposed(view, pdb_prot.slice_arrays(has_coords_indices))
view.zoomTo()
view.show()

Now we get the AFDB structure, slice out the non-null, and overlay

In [ ]:
chain = chain.remap(res.mapping[res.hits[res.hits["pdb_id"]=="7zju"].iloc[0]["mapping_index"]])

In [ ]:
chain.coords.residue_index.unique()  # this corresponds to chain.residues.index

In [ ]:
chain.residues.index

In [ ]:
chain.residues["id"].unique()